# Experiment: Intelligence API 故障巡检

Objective:
- 复现并定位 `INTELLIGENCE_DATA_UNAVAILABLE`、`上游 provider 不可用`、`获取概念板块列表失败` 这条异常链路。
- 一次性跑完 HTTP 接口、FastAPI 网关、AkShare adapter、AkShare 原始函数，判断故障位于哪一层。

使用方式：
- 先启动 `python_services`，再从上到下执行。
- 如需换主题/股票，修改下面配置单元里的 `TEST_THEME` / `FALLBACK_STOCK_CODE` / `FALLBACK_CONCEPT`。


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'python_services').exists():
            return candidate
    fallback = Path(os.getenv('STOCK_SCREENING_REPO_ROOT', 'D:/课外项目/stock-screening-boost')).resolve()
    if (fallback / 'python_services').exists():
        return fallback
    raise RuntimeError('找不到仓库根目录。请在仓库根目录启动 Jupyter，或设置 STOCK_SCREENING_REPO_ROOT。')


REPO_ROOT = find_repo_root()
PYTHON_SERVICES_ROOT = REPO_ROOT / 'python_services'
if str(PYTHON_SERVICES_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SERVICES_ROOT))

BASE_URL = os.getenv('INTELLIGENCE_BASE_URL', 'http://127.0.0.1:8000').rstrip('/')
REQUEST_TIMEOUT_SECONDS = int(os.getenv('INTELLIGENCE_HTTP_TIMEOUT_SECONDS', '30'))
TEST_THEME = os.getenv('INTELLIGENCE_TEST_THEME', '算力').strip()
FALLBACK_CONCEPT = os.getenv('INTELLIGENCE_TEST_CONCEPT', '算力租赁').strip()
FALLBACK_STOCK_CODE = os.getenv('INTELLIGENCE_TEST_STOCK_CODE', '300308').strip()

pd.set_option('display.max_colwidth', 120)


def unwrap_exception_chain(exc: BaseException) -> list[str]:
    chain: list[str] = []
    current: BaseException | None = exc
    seen: set[int] = set()
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        chain.append(f'{type(current).__name__}: {current}')
        if current.__cause__ is not None:
            current = current.__cause__
        elif current.__context__ is not None:
            current = current.__context__
        elif len(getattr(current, 'args', ())) > 1 and isinstance(current.args[1], BaseException):
            current = current.args[1]
        else:
            current = None
    return chain


def preview(value: Any, limit: int = 700) -> str:
    if isinstance(value, pd.DataFrame):
        text = value.head(5).to_string(index=False)
    else:
        try:
            text = json.dumps(value, ensure_ascii=False, indent=2, default=str)
        except TypeError:
            text = repr(value)
    return text if len(text) <= limit else f'{text[:limit]} ...'


def http_probe(label: str, method: str, path: str, *, params: dict[str, Any] | None = None, json_body: dict[str, Any] | None = None) -> dict[str, Any]:
    started = time.perf_counter()
    try:
        response = requests.request(method=method, url=f'{BASE_URL}{path}', params=params, json=json_body, timeout=REQUEST_TIMEOUT_SECONDS)
        elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
        try:
            payload = response.json()
        except Exception:
            payload = response.text
        meta = payload.get('meta', {}) if isinstance(payload, dict) else {}
        error = payload.get('error', {}) if isinstance(payload, dict) else {}
        return {
            'label': label,
            'ok': response.ok,
            'status': response.status_code,
            'elapsed_ms': elapsed_ms,
            'provider': meta.get('provider', ''),
            'error_code': error.get('code', ''),
            'error_message': error.get('message', '') if isinstance(error, dict) else '',
            'payload': payload,
        }
    except Exception as exc:
        return {
            'label': label,
            'ok': False,
            'status': None,
            'elapsed_ms': round((time.perf_counter() - started) * 1000, 2),
            'provider': '',
            'error_code': type(exc).__name__,
            'error_message': ' | '.join(unwrap_exception_chain(exc)),
            'payload': None,
        }


def python_probe(label: str, fn) -> dict[str, Any]:
    started = time.perf_counter()
    try:
        value = fn()
        if isinstance(value, pd.DataFrame):
            summary = f'DataFrame shape={value.shape}'
        elif isinstance(value, list):
            summary = f'list[{len(value)}]'
        elif isinstance(value, dict):
            summary = f'dict keys={list(value.keys())[:8]}'
        else:
            summary = repr(value)[:160]
        return {'label': label, 'ok': True, 'elapsed_ms': round((time.perf_counter() - started) * 1000, 2), 'summary': summary, 'error_message': '', 'value': value}
    except Exception as exc:
        return {'label': label, 'ok': False, 'elapsed_ms': round((time.perf_counter() - started) * 1000, 2), 'summary': '', 'error_message': ' | '.join(unwrap_exception_chain(exc)), 'value': None}


def pick_first_concept(payload: Any) -> str | None:
    data = payload.get('data') if isinstance(payload, dict) and 'data' in payload else payload
    if not isinstance(data, dict):
        return None
    matches = data.get('conceptMatches') or data.get('concepts') or []
    if not matches or not isinstance(matches[0], dict):
        return None
    return str(matches[0].get('name') or matches[0].get('concept') or '').strip() or None


def pick_first_stock(payload: Any) -> str | None:
    data = payload.get('data') if isinstance(payload, dict) and 'data' in payload else payload
    if not isinstance(data, dict):
        return None
    items = data.get('candidates') or data.get('items') or []
    if not items or not isinstance(items[0], dict):
        return None
    return str(items[0].get('stockCode') or items[0].get('code') or '').strip() or None


config = {
    'REPO_ROOT': str(REPO_ROOT),
    'PYTHON_SERVICES_ROOT': str(PYTHON_SERVICES_ROOT),
    'BASE_URL': BASE_URL,
    'REQUEST_TIMEOUT_SECONDS': REQUEST_TIMEOUT_SECONDS,
    'TEST_THEME': TEST_THEME,
    'FALLBACK_CONCEPT': FALLBACK_CONCEPT,
    'FALLBACK_STOCK_CODE': FALLBACK_STOCK_CODE,
}
config


In [ ]:
basic_results = [
    http_probe('root', 'GET', '/'),
    http_probe('health', 'GET', '/health'),
    http_probe('admin_metrics', 'GET', '/api/admin/metrics'),
]

v1_theme_results = [
    http_probe('v1_concepts', 'GET', f'/api/v1/intelligence/themes/{TEST_THEME}/concepts', params={'limit': 3}),
    http_probe('v1_candidates', 'GET', f'/api/v1/market/themes/{TEST_THEME}/candidates', params={'limit': 3}),
    http_probe('v1_news', 'GET', f'/api/v1/intelligence/themes/{TEST_THEME}/news', params={'days': 7, 'limit': 5}),
]

v1_by_label = {item['label']: item for item in v1_theme_results}
SELECTED_CONCEPT = pick_first_concept(v1_by_label['v1_concepts']['payload']) or FALLBACK_CONCEPT
SELECTED_STOCK_CODE = pick_first_stock(v1_by_label['v1_candidates']['payload']) or FALLBACK_STOCK_CODE
BATCH_CODES = list(dict.fromkeys([SELECTED_STOCK_CODE, FALLBACK_STOCK_CODE]))

v1_stock_results = [
    http_probe('v1_market_stock', 'GET', f'/api/v1/market/stocks/{SELECTED_STOCK_CODE}'),
    http_probe('v1_market_history', 'GET', f'/api/v1/market/stocks/{SELECTED_STOCK_CODE}/history', params={'indicator': 'ROE', 'years': 3}),
    http_probe('v1_evidence', 'GET', f'/api/v1/intelligence/stocks/{SELECTED_STOCK_CODE}/evidence', params={'concept': SELECTED_CONCEPT}),
    http_probe('v1_evidence_batch', 'POST', '/api/v1/intelligence/stocks/evidence/batch', json_body={'stockCodes': BATCH_CODES, 'concept': SELECTED_CONCEPT}),
    http_probe('v1_research_pack', 'GET', f'/api/v1/intelligence/stocks/{SELECTED_STOCK_CODE}/research-pack', params={'concept': SELECTED_CONCEPT}),
]

legacy_results = [
    http_probe('legacy_concepts_match', 'GET', '/api/intelligence/concepts/match', params={'theme': TEST_THEME, 'limit': 3}),
    http_probe('legacy_candidates', 'GET', '/api/intelligence/candidates', params={'theme': TEST_THEME, 'limit': 3}),
    http_probe('legacy_news', 'GET', '/api/intelligence/news', params={'theme': TEST_THEME, 'days': 7, 'limit': 5}),
    http_probe('legacy_evidence', 'GET', f'/api/intelligence/evidence/{SELECTED_STOCK_CODE}', params={'concept': SELECTED_CONCEPT}),
    http_probe('legacy_evidence_batch', 'POST', '/api/intelligence/evidence/batch', json_body={'stockCodes': BATCH_CODES, 'concept': SELECTED_CONCEPT}),
]

admin_results = [
    http_probe('admin_refresh_concepts', 'POST', '/api/admin/jobs/refresh-concepts', json_body={'limit': 20}),
]

http_results = basic_results + v1_theme_results + v1_stock_results + legacy_results + admin_results
http_df = pd.DataFrame([
    {
        'label': item['label'],
        'ok': item['ok'],
        'status': item['status'],
        'elapsed_ms': item['elapsed_ms'],
        'provider': item['provider'],
        'error_code': item['error_code'],
        'error_message': item['error_message'],
    }
    for item in http_results
])

display(http_df)
display(Markdown(f'**后续本地 probe 使用** concept=`{SELECTED_CONCEPT}`，stock=`{SELECTED_STOCK_CODE}`'))
print('v1_concepts preview:')
print(preview(v1_by_label['v1_concepts']['payload']))
print('\nadmin_refresh_concepts preview:')
print(preview(admin_results[0]['payload']))


In [ ]:
local_import_error = None
local_results: list[dict[str, Any]] = []

try:
    import akshare as ak
    from app.providers.akshare.client import AkShareProviderClient
    from app.services.akshare_adapter import AkShareAdapter
    from app.services.intelligence_data_adapter import IntelligenceDataAdapter

    provider_client = AkShareProviderClient()
    print(f"akshare version: {getattr(ak, '__version__', 'unknown')}")
except Exception as exc:
    local_import_error = exc
    print('本地模块导入失败：')
    print(' | '.join(unwrap_exception_chain(exc)))

if local_import_error is None:
    local_results = [
        python_probe('AkShareAdapter.get_concept_catalog_frame', AkShareAdapter.get_concept_catalog_frame),
        python_probe('AkShareProviderClient.get_concept_catalog', provider_client.get_concept_catalog),
        python_probe('IntelligenceDataAdapter.match_theme_concepts', lambda: IntelligenceDataAdapter.match_theme_concepts(TEST_THEME, limit=3)),
        python_probe('IntelligenceDataAdapter.get_candidates_strict', lambda: IntelligenceDataAdapter.get_candidates_strict(TEST_THEME, limit=3)),
        python_probe('IntelligenceDataAdapter.get_theme_news_strict', lambda: IntelligenceDataAdapter.get_theme_news_strict(TEST_THEME, days=7, limit=5)),
        python_probe('IntelligenceDataAdapter.get_company_evidence_strict', lambda: IntelligenceDataAdapter.get_company_evidence_strict(SELECTED_STOCK_CODE, SELECTED_CONCEPT)),
        python_probe('IntelligenceDataAdapter.get_company_research_pack_strict', lambda: IntelligenceDataAdapter.get_company_research_pack_strict(SELECTED_STOCK_CODE, SELECTED_CONCEPT)),
    ]

local_df = pd.DataFrame([
    {
        'label': item['label'],
        'ok': item['ok'],
        'elapsed_ms': item['elapsed_ms'],
        'summary': item['summary'],
        'error_message': item['error_message'],
    }
    for item in local_results
])
local_df


In [ ]:
raw_results: list[dict[str, Any]] = []

def find_probe(results: list[dict[str, Any]], label: str) -> dict[str, Any] | None:
    for item in results:
        if item['label'] == label:
            return item
    return None


def resolve_concept_symbol(frame: pd.DataFrame, concept_name: str) -> str | None:
    if frame.empty:
        return None
    text_columns = [column for column in frame.columns if '名称' in str(column) or '概念' in str(column)]
    code_columns = [column for column in frame.columns if '代码' in str(column)]
    for _, row in frame.iterrows():
        names = [str(row.get(column) or '').strip() for column in text_columns]
        if concept_name in names:
            for column in code_columns:
                value = str(row.get(column) or '').strip()
                if value:
                    return value
            return next((name for name in names if name), None)
    first_row = frame.iloc[0]
    for column in code_columns + text_columns:
        value = str(first_row.get(column) or '').strip()
        if value:
            return value
    return None

if local_import_error is None:
    raw_results.append(python_probe('ak.stock_zh_a_spot_em', lambda: ak.stock_zh_a_spot_em().head(5)))
    raw_catalog = python_probe('ak.stock_board_concept_name_em', ak.stock_board_concept_name_em)
    raw_results.append(raw_catalog)
    concept_frame = raw_catalog['value'] if raw_catalog['ok'] else pd.DataFrame()
    concept_symbol = resolve_concept_symbol(concept_frame, SELECTED_CONCEPT) if isinstance(concept_frame, pd.DataFrame) else None
    if concept_symbol:
        raw_results.append(python_probe(f'ak.stock_board_concept_cons_em({concept_symbol})', lambda symbol=concept_symbol: ak.stock_board_concept_cons_em(symbol=symbol).head(10)))
    else:
        raw_results.append({'label': 'ak.stock_board_concept_cons_em(SKIPPED)', 'ok': False, 'elapsed_ms': 0.0, 'summary': '', 'error_message': '未拿到可用的 concept symbol，已跳过。', 'value': None})

raw_df = pd.DataFrame([
    {
        'label': item['label'],
        'ok': item['ok'],
        'elapsed_ms': item['elapsed_ms'],
        'summary': item.get('summary', ''),
        'error_message': item.get('error_message', ''),
    }
    for item in raw_results
])
display(raw_df)

http_concepts = v1_by_label['v1_concepts']
adapter_catalog = find_probe(local_results, 'AkShareAdapter.get_concept_catalog_frame')
raw_catalog = find_probe(raw_results, 'ak.stock_board_concept_name_em')

suspected_layer = '暂未判定'
conclusion_lines: list[str] = []
if http_concepts['ok']:
    suspected_layer = '问题未复现 / 间歇性故障'
    conclusion_lines.append('v1 概念接口本次可用，建议多跑几次或更换主题复测。')
else:
    conclusion_lines.append(f"HTTP 层报错：{http_concepts['error_message'] or http_concepts['status']}")
    if local_import_error is not None:
        suspected_layer = 'HTTP 层失败，但本地依赖未就绪'
        conclusion_lines.append('本地 app / akshare 模块未成功导入，无法继续缩小范围。')
    elif raw_catalog and not raw_catalog['ok']:
        suspected_layer = 'AkShare 概念板块上游 / 网络链路'
        conclusion_lines.append('直接调用 `ak.stock_board_concept_name_em()` 也失败，说明不是 FastAPI 包装层独有问题。')
    elif adapter_catalog and not adapter_catalog['ok']:
        suspected_layer = '本地 adapter / provider'
        conclusion_lines.append('AkShare 原始调用成功，但 adapter/provider 失败，优先检查本地封装与缓存逻辑。')
    else:
        suspected_layer = 'FastAPI gateway / HTTP 层'
        conclusion_lines.append('本地 adapter 与 AkShare 看起来正常，优先检查 gateway / 请求参数 / 运行环境。')

summary = {
    'suspected_layer': suspected_layer,
    'theme': TEST_THEME,
    'stock_code': SELECTED_STOCK_CODE,
    'concept': SELECTED_CONCEPT,
    'http_concepts_ok': http_concepts['ok'],
    'local_import_error': None if local_import_error is None else ' | '.join(unwrap_exception_chain(local_import_error)),
    'adapter_catalog_error': None if adapter_catalog is None else adapter_catalog['error_message'] or None,
    'raw_catalog_error': None if raw_catalog is None else raw_catalog['error_message'] or None,
}

display(Markdown('### Notebook summary'))
for line in conclusion_lines:
    print(f'- {line}')
summary


## Next steps

- 如果 `ak.stock_board_concept_name_em()` 直接失败：优先怀疑 AkShare 对应源站临时不可用、网络代理/防火墙、或需要换时间重试。
- 如果只有 `/api/v1/intelligence/themes/{theme}/concepts` 失败：检查 FastAPI 进程日志、gateway cache、以及 `python_services/app/services/intelligence_data_adapter.py`。
- 如果 `refresh-concepts` 失败：它会触发概念目录与成分股预热，是复现这次 503 最直接的一步。
